In [1]:
geo_path = '/eos/user/j/jrimmer/Geometry'
import sys
sys.path.insert(0, geo_path)
sys.path.insert(0, "../")
sys.path.insert(0, "../scripts")
sys.path.insert(0, "../LicketyFit")
from Geometry.Device import Device

from LicketyFit.Event import *
from LicketyFit.PMT import *

from LicketyFit.Emitter import *

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import uproot, json, awkward as ak
from read_sim_data import *
from plot_event import *

with open('/eos/user/j/jrimmer/SWAN_projects/beam/LicketyFit2/tables/other_mpmt_info_v2.dict', 'rb') as f:
    mpmt_info = pickle.load(f)
    

In [2]:
# =============================================================================
# Single-event LF_multiParticles notebook fitter
#
# Supports:
#
#   FIT_MODE = "full_length"
#       7-parameter fit:
#           x0, y0, z0, cx, cy, length, t0
#
#       length = full visible track length / dE/dx range-to-threshold [mm]
#       ke0    = inferred from length using the particle range table [MeV]
#
#   FIT_MODE = "absorption"
#       8-parameter fit:
#           x0, y0, z0, cx, cy, visible_length, full_range, t0
#
#       visible_length = visible primary-Cherenkov length before absorption/cutoff [mm]
#       full_range     = dE/dx-only range-to-threshold without absorption [mm]
#       ke0            = inferred from full_range using the particle range table [MeV]
#
# Input event format:
#   event_array[:, 0] = WCTE PMT ID = slot*100 + pmt_position
#   event_array[:, 1] = charge
#   event_array[:, 2] = time
#
# Main output:
#   result["pmt_ids"]
#   result["exp_pes"]
#   result["exp_ts"]
#   result["obs_pes"]
#   result["obs_ts"]
# =============================================================================

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import uproot
from iminuit import Minuit
# Debug output.
RETURN_FULL_SEED_SCAN = False
RETURN_TOP_N_SEEDS = 10

# =============================================================================
# 1. Path setup
# =============================================================================
WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent

LICKETYFIT_DIR = PROJECT_ROOT / "LicketyFit"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
TABLES_DIR = PROJECT_ROOT / "tables"

GEOMETRY_PATH = Path(os.environ.get("GEOMETRY_PATH", "/eos/user/j/jrimmer/Geometry"))
GEOMETRY_FILE = Path(
    os.environ.get(
        "WCTE_GEOMETRY_FILE",
        str(GEOMETRY_PATH / "examples" / "wcte_bldg157.geo"),
    )
)

for p in [PROJECT_ROOT, LICKETYFIT_DIR, SCRIPTS_DIR, TABLES_DIR, GEOMETRY_PATH]:
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["LF_TABLE_DIR"] = str(TABLES_DIR)
os.environ["LF_MULTIPARTICLES_TABLE_DIR"] = str(TABLES_DIR)

# Clear stale notebook imports after changing files on disk.
for mod in list(sys.modules):
    if (
        mod == "particle_cherenkov_model"
        or mod == "LicketyFit.particle_cherenkov_model"
        or mod == "particle_range_lookup"
        or mod == "n_model_wrapper"
        or mod == "LicketyFit.Emitter"
    ):
        del sys.modules[mod]


# =============================================================================
# 2. Imports
# =============================================================================
from Geometry.Device import Device
from LicketyFit.Event import Event
from LicketyFit.PMT import PMT
from LicketyFit.Emitter import Emitter

from particle_cherenkov_model import (
    get_energy_distance_tables,
    set_active_particle,
    canonical_particle_name,
    particle_mass_mev,
    cherenkov_threshold_kinetic_mev,
)

from particle_range_lookup import ParticleRangeLookup


# =============================================================================
# 3. User settings
# =============================================================================

FIT_PARTICLE = "muon"          # "muon", "pion", "kaon", "proton"
FIT_MODE = "full_length"      # "full_length" or "absorption"

# Geometry/PMT behavior.
PMT_PLACEMENT = "est"         # "est" for real data, "design" for WCSim-like checks
PE_SCALE = 143.0              # real-data charge scale; use 1.0 for WCSim-style arrays

# Time handling.
APPLY_PEAK_TIME_WINDOW = True
PEAK_HIST_MIN_NS = 0.0
PEAK_HIST_MAX_NS = 4000.0
PEAK_HIST_BIN_WIDTH_NS = 1.0
N_BINS_BEFORE_PEAK = 20
N_BINS_AFTER_PEAK = 5

SHIFT_TIMES = True
N_EARLIEST_FOR_T0 = 10

# Likelihood.
USE_CHARGE_LIKELIHOOD = True
USE_TIMING_LIKELIHOOD = False
USE_T0_PRIOR = False

M_STRAT = 1
NCALL_SIMPLEX = 30000
NCALL_MIGRAD = 70000
USE_MIGRAD_FIRST = False
T0_LIMITS = (-8.0, 8.0)

# Detector masking.
DEFAULT_INACTIVE_SLOTS = [27, 32, 45, 74, 77, 79, 85, 91, 99, 9, 67]
INACTIVE_SLOTS = DEFAULT_INACTIVE_SLOTS
INACTIVE_SLOTS_SET = set(int(s) for s in INACTIVE_SLOTS)

USE_GOOD_WCTE_PMTS = True
ALLOW_MISSING_GOOD_PMTS = True
CONFIG_ROOT_FILE = None
# Example:
# CONFIG_ROOT_FILE = "/eos/experiment/wcte/data/2025_commissioning/processed_offline_data/production_v1_0/2079/WCTE_merged_production_R2079.root"

OTHER_MPMT_INFO_PATH = TABLES_DIR / "other_mpmt_info_v2.dict"

# Ring mask.
OUTER_RING = np.array([0, 7, 19, 34, 50, 66, 82, 83, 105, 94, 95, 71, 72, 56, 40, 24, 11, 3, 18])
INNER_RING = np.array([1, 8, 35, 51, 67, 84, 69, 70, 55, 39, 23, 10, 2, 20, 36, 52, 68, 53, 54, 38, 22, 21, 37, 9])
ALL_RING = np.arange(0, 106)
RING_MASK_MODE = "both"       # "none", "pes", "ts", "both"

# Seed grid.
FAST_SEED_X0 = [-50.0, 0.0, 50.0]
FAST_SEED_Y0 = [-50.0, 0.0, 50.0]
FAST_SEED_Z0 = [-1500.0, -1400.0, -1350.0, -1300.0, -1200.0, -1100.0, -1000.0]

FAST_SEED_LENGTHS = [
    100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0,
    500.0, 700.0, 900.0, 1100.0, 1300.0, 1400.0, 1500.0,
    1700.0, 1900.0,
]

FAST_SEED_VISIBLE_LENGTHS = FAST_SEED_LENGTHS.copy()

FAST_SEED_KE0_MEV = [600.0, 800.0, 1000.0, 1200.0, 1500.0, 2000.0]
FAST_SEED_FULL_RANGES_MM = [300.0, 600.0, 1000.0, 1500.0, 2200.0, 3000.0]

FAST_SEED_DIRECTIONS = [
    (0.0, 0.0),
    (0.04, 0.0),
    (-0.04, 0.0),
    (0.0, 0.04),
    (0.0, -0.04),
]

FAST_SEED_FULL_CARTESIAN = False




# =============================================================================
# 4. Mode helpers
# =============================================================================
def canonical_fit_mode(mode):
    key = str(mode).strip().lower()

    if key in {"full_length", "full-length", "full", "threshold", "range", "csda", "old", "7param", "7_parameter"}:
        return "full_length"

    if key in {"absorption", "absorbed", "abrupt", "abrupt_8param", "interaction", "truncated", "8param", "8_parameter"}:
        return "absorption"

    raise ValueError("FIT_MODE must be 'full_length' or 'absorption'")


def is_absorption_mode(fit_mode):
    return canonical_fit_mode(fit_mode) == "absorption"


def emitter_track_end_mode_for_fit_mode(fit_mode):
    return "abrupt" if is_absorption_mode(fit_mode) else "threshold"


# =============================================================================
# 5. mPMT / good-PMT helpers
# =============================================================================
def load_good_wcte_pmts(config_root_file=None):
    if not USE_GOOD_WCTE_PMTS:
        good = []
        for slot in range(106):
            if slot in INACTIVE_SLOTS_SET:
                continue
            for pmt_pos in range(19):
                good.append(slot * 100 + pmt_pos)
        return set(good)

    if config_root_file is None:
        if ALLOW_MISSING_GOOD_PMTS:
            print("No CONFIG_ROOT_FILE set; using all non-inactive PMTs.")
            good = []
            for slot in range(106):
                if slot in INACTIVE_SLOTS_SET:
                    continue
                for pmt_pos in range(19):
                    good.append(slot * 100 + pmt_pos)
            return set(good)
        raise ValueError("CONFIG_ROOT_FILE is required when USE_GOOD_WCTE_PMTS=True")

    try:
        with uproot.open(config_root_file) as f:
            t_c = f["Configuration"]
            arr_config = t_c.arrays(library="ak")
        good = np.asarray(arr_config["good_wcte_pmts"][0], dtype=int)
        print("Loaded GOOD_WCTE_PMTS from:", config_root_file)
        return set(good.tolist())
    except Exception as exc:
        if ALLOW_MISSING_GOOD_PMTS:
            print("WARNING: could not load GOOD_WCTE_PMTS; using all non-inactive PMTs.")
            print("Reason:", repr(exc))
            good = []
            for slot in range(106):
                if slot in INACTIVE_SLOTS_SET:
                    continue
                for pmt_pos in range(19):
                    good.append(slot * 100 + pmt_pos)
            return set(good)
        raise


def load_mpmt_info(path=OTHER_MPMT_INFO_PATH):
    path = Path(path)
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    print("No other_mpmt_info_v2.dict found; using empty mPMT types.")
    return {}


def get_mpmt_slot_type(mpmt_slots, mpmt_info):
    slot_type = []

    for slot in mpmt_slots:
        slot = int(slot)
        try:
            if mpmt_info[slot]["mpmt_site"] == "TRI":
                if mpmt_info[slot]["mpmt_type"] == "In-situ":
                    slot_type.append("tri_insitu")
                else:
                    slot_type.append("tri_exsitu")
            else:
                if mpmt_info[slot]["mpmt_type"] == "In-situ":
                    slot_type.append("wut_insitu")
                else:
                    slot_type.append("wut_exsitu")
        except Exception:
            slot_type.append("empty")

    return slot_type


# =============================================================================
# 6. Event / observable helpers
# =============================================================================
def apply_peak_time_window_to_hit_array(event_array):
    event = np.asarray(event_array, dtype=np.float64)
    if event.ndim != 2 or event.shape[1] < 3:
        raise ValueError("event_array must have shape (N_hits, >=3): [pmt_id, charge, time]")

    if event.size == 0:
        return event[:, :3]

    times = np.asarray(event[:, 2], dtype=np.float64)
    times = times[np.isfinite(times)]

    if times.size == 0:
        return event[:, :3]

    bins = np.arange(PEAK_HIST_MIN_NS, PEAK_HIST_MAX_NS + PEAK_HIST_BIN_WIDTH_NS, PEAK_HIST_BIN_WIDTH_NS)
    counts, edges = np.histogram(times, bins=bins)

    if counts.size == 0 or np.max(counts) == 0:
        return event[:, :3]

    max_idx = int(np.argmax(counts))
    lo_idx = max(0, max_idx - N_BINS_BEFORE_PEAK)
    hi_idx = min(len(edges) - 1, max_idx + N_BINS_AFTER_PEAK)

    min_time = edges[lo_idx]
    max_time = edges[hi_idx]

    keep = (event[:, 2] > min_time) & (event[:, 2] < max_time)
    return event[keep, :3]


def hit_array_to_event(
    event_array,
    WCD,
    good_wcte_pmts_set,
    *,
    n_mpmt_total=106,
    shift_times=True,
    n_earliest_for_t0=10,
):
    """
    Convert one hit array into a LicketyFit Event.

    event_array columns:
        0: WCTE PMT ID = slot*100 + pmt_position
        1: charge
        2: time
    """
    event_array = np.asarray(event_array, dtype=np.float64)

    if event_array.ndim != 2 or event_array.shape[1] < 3:
        raise ValueError("event_array must be a 2D array with columns [pmt_id, charge, time]")

    vw = 223.0598645833333  # mm/ns

    ev = Event(0, 0, n_mpmt_total)
    ev.set_mpmt_status(list(range(n_mpmt_total)), False)

    active_wcte_pmt_ids = []

    for slot in range(n_mpmt_total):
        if slot in INACTIVE_SLOTS_SET:
            continue

        slot_has_good_pmt = False

        for pmt_pos in range(ev.npmt_per_mpmt):
            wcte_pmt = int(slot * 100 + pmt_pos)

            if wcte_pmt in good_wcte_pmts_set:
                ev.set_pmt_status(slot, [pmt_pos], True)
                slot_has_good_pmt = True
                active_wcte_pmt_ids.append(wcte_pmt)

        if slot_has_good_pmt:
            ev.set_mpmt_status([slot], True)

    skipped = 0

    for row in event_array[:, :3]:
        wcte_pmt = int(row[0])
        charge = float(row[1])
        time = float(row[2])

        slot = int(wcte_pmt // 100)
        pmt_pos = int(wcte_pmt % 100)

        if slot < 0 or slot >= ev.n_mpmt:
            skipped += 1
            continue
        if pmt_pos < 0 or pmt_pos >= ev.npmt_per_mpmt:
            skipped += 1
            continue
        if not ev.mpmt_status[slot]:
            continue
        if not ev.pmt_status[slot][pmt_pos]:
            continue

        ev.hit_charges[slot][pmt_pos].append(charge)
        ev.hit_times[slot][pmt_pos].append(time)

    if skipped:
        print(f"Skipped {skipped} hits with invalid PMT IDs.")

    if shift_times:
        bp_loc = np.array([0.0, 0.0, -1350.0])
        early_hits = []

        for i_mpmt in range(ev.n_mpmt):
            if not ev.mpmt_status[i_mpmt]:
                continue

            mpmt = WCD.mpmts[i_mpmt]
            if mpmt is None:
                continue

            for i_pmt in range(ev.npmt_per_mpmt):
                if not ev.pmt_status[i_mpmt][i_pmt]:
                    continue
                if len(ev.hit_times[i_mpmt][i_pmt]) == 0:
                    continue

                pmt = mpmt.pmts[i_pmt]
                if pmt is None:
                    continue

                try:
                    placement = pmt.get_placement(PMT_PLACEMENT, WCD)
                except TypeError:
                    placement = pmt.get_placement(PMT_PLACEMENT)

                pmt_loc = np.asarray(placement["location"], dtype=np.float64)
                r = np.linalg.norm(pmt_loc - bp_loc)

                for t in ev.hit_times[i_mpmt][i_pmt]:
                    early_hits.append({
                        "time": float(t),
                        "t0_est": float(t) - r / vw,
                    })

        if len(early_hits) > 0:
            early_hits = sorted(early_hits, key=lambda x: x["time"])
            n_use = min(n_earliest_for_t0, len(early_hits))
            time_offset = np.median([hit["t0_est"] for hit in early_hits[:n_use]])

            for i_mpmt in range(ev.n_mpmt):
                for i_pmt in range(ev.npmt_per_mpmt):
                    ev.hit_times[i_mpmt][i_pmt] = [
                        t - time_offset for t in ev.hit_times[i_mpmt][i_pmt]
                    ]

            ev.global_time_offset = float(time_offset)

    return ev, np.asarray(active_wcte_pmt_ids, dtype=int)


def get_pmt_placements_with_ids(event, WCD, place_info):
    p_locations = []
    direction_zs = []
    mpmt_slots = []
    pmt_ids = []

    for i_mpmt in range(event.n_mpmt):
        if not event.mpmt_status[i_mpmt]:
            continue

        mpmt = WCD.mpmts[i_mpmt]
        if mpmt is None:
            continue

        for i_pmt in range(event.npmt_per_mpmt):
            if not event.pmt_status[i_mpmt][i_pmt]:
                continue

            pmt = mpmt.pmts[i_pmt]
            if pmt is None:
                continue

            try:
                placement = pmt.get_placement(place_info, WCD)
            except TypeError:
                placement = pmt.get_placement(place_info)

            p_locations.append(np.asarray(placement["location"], dtype=np.float64))
            direction_zs.append(np.asarray(placement["direction_z"], dtype=np.float64))
            mpmt_slots.append(int(i_mpmt))
            pmt_ids.append(int(i_mpmt * 100 + i_pmt))

    return (
        np.asarray(p_locations, dtype=np.float64),
        np.asarray(direction_zs, dtype=np.float64),
        np.asarray(mpmt_slots, dtype=np.int64),
        np.asarray(pmt_ids, dtype=np.int64),
    )


def build_observables_from_event(ev, pe_scale=143.0):
    obs_pes = []
    obs_ts = []

    for i_mpmt in range(ev.n_mpmt):
        if not ev.mpmt_status[i_mpmt]:
            continue

        for i_pmt in range(ev.npmt_per_mpmt):
            if not ev.pmt_status[i_mpmt][i_pmt]:
                continue

            q = np.asarray(ev.hit_charges[i_mpmt][i_pmt], dtype=np.float64)
            t = np.asarray(ev.hit_times[i_mpmt][i_pmt], dtype=np.float64)

            if q.size == 0:
                obs_pes.append(0.0)
                obs_ts.append(np.nan)
            else:
                obs_pes.append(float(np.sum(q)) / pe_scale)
                obs_ts.append(float(np.sum(q * t) / np.sum(q)))

    return np.asarray(obs_pes, dtype=np.float64), np.asarray(obs_ts, dtype=np.float64)


def apply_ring_mask_to_observables(obs_pes, obs_ts, ring_keep_mask, mode="both"):
    obs_pes = np.asarray(obs_pes, dtype=np.float64).copy()
    obs_ts = np.asarray(obs_ts, dtype=np.float64).copy()

    if mode not in {"none", "pes", "ts", "both"}:
        raise ValueError("RING_MASK_MODE must be one of: none, pes, ts, both")

    if mode in {"pes", "both"}:
        obs_pes[~ring_keep_mask] = 0.0

    if mode in {"ts", "both"}:
        obs_ts[~ring_keep_mask] = np.nan

    return obs_pes, obs_ts


def unit_direction_from_cxcy(cx, cy):
    cx = float(cx)
    cy = float(cy)

    cz2 = 1.0 - cx * cx - cy * cy

    if cz2 <= 0.0:
        raise ValueError(f"Invalid direction: cx^2 + cy^2 = {cx*cx + cy*cy} >= 1")

    return (cx, cy, float(np.sqrt(cz2)))


# =============================================================================
# 7. Seed grid
# =============================================================================
def build_sparse_geometry_variants():
    variants = []

    for x0 in FAST_SEED_X0:
        variants.append({"x0": float(x0), "y0": 0.0, "cx": 0.0, "cy": 0.0})

    for y0 in FAST_SEED_Y0:
        variants.append({"x0": 0.0, "y0": float(y0), "cx": 0.0, "cy": 0.0})

    for cx, cy in FAST_SEED_DIRECTIONS:
        variants.append({"x0": 0.0, "y0": 0.0, "cx": float(cx), "cy": float(cy)})

    unique = []
    seen = set()

    for v in variants:
        sig = (v["x0"], v["y0"], v["cx"], v["cy"])
        if sig not in seen:
            seen.add(sig)
            unique.append(v)

    return unique


def build_full_range_seed_values(range_lookup):
    values = []

    for ke0 in FAST_SEED_KE0_MEV:
        if ke0 <= range_lookup.threshold_mev:
            continue

        try:
            r = range_lookup.energy_to_range_mm(float(ke0))
        except Exception:
            continue

        if np.isfinite(r) and r > 0:
            values.append(float(r))

    values.extend(float(r) for r in FAST_SEED_FULL_RANGES_MM)

    max_r = float(range_lookup.overall_distances_mm[-1])
    values = [r for r in values if np.isfinite(r) and 0.0 < r <= max_r]

    unique = []
    seen = set()

    for r in values:
        sig = round(float(r), 6)
        if sig not in seen:
            seen.add(sig)
            unique.append(float(r))

    return unique


def build_fast_seed_grid(range_lookup, fit_mode):
    fit_mode = canonical_fit_mode(fit_mode)
    geometry_variants = build_sparse_geometry_variants()
    seeds = []

    if fit_mode == "full_length":
        max_r = float(range_lookup.overall_distances_mm[-1])
        length_seeds = [float(x) for x in FAST_SEED_LENGTHS if 0.0 < float(x) <= max_r]

        if FAST_SEED_FULL_CARTESIAN:
            for x0 in FAST_SEED_X0:
                for y0 in FAST_SEED_Y0:
                    for z0 in FAST_SEED_Z0:
                        for length in length_seeds:
                            for cx, cy in FAST_SEED_DIRECTIONS:
                                seeds.append({
                                    "x0": float(x0),
                                    "y0": float(y0),
                                    "z0": float(z0),
                                    "cx": float(cx),
                                    "cy": float(cy),
                                    "length": float(length),
                                    "t0": 0.0,
                                })
        else:
            for z0 in FAST_SEED_Z0:
                for length in length_seeds:
                    for geom in geometry_variants:
                        seeds.append({
                            "x0": float(geom["x0"]),
                            "y0": float(geom["y0"]),
                            "z0": float(z0),
                            "cx": float(geom["cx"]),
                            "cy": float(geom["cy"]),
                            "length": float(length),
                            "t0": 0.0,
                        })

        keys = ("x0", "y0", "z0", "cx", "cy", "length", "t0")

    else:
        full_range_seeds = build_full_range_seed_values(range_lookup)

        if FAST_SEED_FULL_CARTESIAN:
            for x0 in FAST_SEED_X0:
                for y0 in FAST_SEED_Y0:
                    for z0 in FAST_SEED_Z0:
                        for visible_length in FAST_SEED_VISIBLE_LENGTHS:
                            for full_range in full_range_seeds:
                                if visible_length > full_range:
                                    continue
                                for cx, cy in FAST_SEED_DIRECTIONS:
                                    seeds.append({
                                        "x0": float(x0),
                                        "y0": float(y0),
                                        "z0": float(z0),
                                        "cx": float(cx),
                                        "cy": float(cy),
                                        "visible_length": float(visible_length),
                                        "full_range": float(full_range),
                                        "t0": 0.0,
                                    })
        else:
            for z0 in FAST_SEED_Z0:
                for visible_length in FAST_SEED_VISIBLE_LENGTHS:
                    for full_range in full_range_seeds:
                        if visible_length > full_range:
                            continue
                        for geom in geometry_variants:
                            seeds.append({
                                "x0": float(geom["x0"]),
                                "y0": float(geom["y0"]),
                                "z0": float(z0),
                                "cx": float(geom["cx"]),
                                "cy": float(geom["cy"]),
                                "visible_length": float(visible_length),
                                "full_range": float(full_range),
                                "t0": 0.0,
                            })

        keys = ("x0", "y0", "z0", "cx", "cy", "visible_length", "full_range", "t0")

    unique = []
    seen = set()

    for seed in seeds:
        sig = tuple(float(seed[k]) for k in keys)
        if sig not in seen:
            seen.add(sig)
            unique.append(seed)

    return unique


# =============================================================================
# 8. Likelihood
# =============================================================================
def get_t0_prior_sigma(obs_pes, obs_ts):
    n_timed = np.count_nonzero(np.isfinite(obs_ts))
    total_pe = np.sum(obs_pes)

    if (n_timed < 250) or (total_pe < 300):
        return 0.1
    elif (n_timed < 275) or (total_pe < 350):
        return 0.2
    elif (n_timed < 300) or (total_pe < 400):
        return 0.3
    elif (n_timed < 325) or (total_pe < 450):
        return 0.4
    elif (n_timed < 350) or (total_pe < 500):
        return 0.5
    elif (n_timed < 375) or (total_pe < 550):
        return 0.6
    elif (n_timed < 400) or (total_pe < 600):
        return 0.7
    elif (n_timed < 425) or (total_pe < 650):
        return 0.8
    elif (n_timed < 450) or (total_pe < 700):
        return 1.0
    elif (n_timed < 475) or (total_pe < 750):
        return 1.2
    elif (n_timed < 500) or (total_pe < 800):
        return 1.4
    elif (n_timed < 525) or (total_pe < 850):
        return 1.6
    elif (n_timed < 550) or (total_pe < 900):
        return 1.8
    return 2.0


def get_timing_only_nll(exp_pes, obs_pes, exp_ts, obs_ts, pmt_model):
    exp_pes = np.asarray(exp_pes, dtype=np.float64)
    obs_pes = np.asarray(obs_pes, dtype=np.float64)
    exp_ts = np.asarray(exp_ts, dtype=np.float64)
    obs_ts = np.asarray(obs_ts, dtype=np.float64)

    mask = (
        (exp_pes > 0.0)
        & (obs_pes > 0.0)
        & np.isfinite(exp_ts)
        & np.isfinite(obs_ts)
    )

    if not np.any(mask):
        return 1e30

    sigma_t = pmt_model.single_pe_time_std / np.sqrt(obs_pes[mask])
    dt = (obs_ts[mask] - exp_ts[mask]) / sigma_t

    return float(0.5 * np.sum(dt * dt))


def evaluate_pmt_nll(exp_pes, obs_pes, exp_ts, obs_ts, pmt_model):
    if USE_CHARGE_LIKELIHOOD and USE_TIMING_LIKELIHOOD:
        return pmt_model.get_neg_log_likelihood_npe_t(exp_pes, obs_pes, exp_ts, obs_ts)

    if USE_CHARGE_LIKELIHOOD:
        return pmt_model.get_neg_log_likelihood_npe(exp_pes, obs_pes)

    return get_timing_only_nll(exp_pes, obs_pes, exp_ts, obs_ts, pmt_model)


def evaluate_neg_log_likelihood(
    fit_mode,
    obs_pes,
    obs_ts,
    emitter,
    mpmt_types,
    p_locations,
    direction_zs,
    WCD,
    pmt_model,
    range_lookup,
    params,
):
    fit_mode = canonical_fit_mode(fit_mode)

    x0 = float(params["x0"])
    y0 = float(params["y0"])
    z0 = float(params["z0"])
    cx = float(params["cx"])
    cy = float(params["cy"])
    t0 = float(params["t0"])

    cz2 = 1.0 - cx * cx - cy * cy
    if cz2 <= 0.0:
        return 1e30

    if fit_mode == "absorption":
        visible_length = float(params["visible_length"])
        full_range = float(params["full_range"])

        if not np.isfinite(visible_length) or not np.isfinite(full_range):
            return 1e30
        if visible_length < 0.0 or full_range <= 0.0:
            return 1e30
        if visible_length > full_range:
            return 1e30
        if full_range > float(range_lookup.overall_distances_mm[-1]):
            return 1e30

        ke0 = float(range_lookup.range_mm_to_energy(full_range))
        if (not np.isfinite(ke0)) or ke0 <= range_lookup.threshold_mev:
            return 1e30

        emitter.fixed_initial_KE = ke0
        track_length_for_emission = visible_length

    else:
        length = float(params["length"])

        if not np.isfinite(length) or length < 0.0:
            return 1e30
        if length > float(range_lookup.overall_distances_mm[-1]):
            return 1e30

        emitter.fixed_initial_KE = None
        track_length_for_emission = length

    emitter.start_coord = (x0, y0, z0)
    emitter.starting_time = t0
    emitter.direction = (cx, cy, float(np.sqrt(cz2)))

    init_ke = emitter.refresh_kinematics_from_length(track_length_for_emission)

    if hasattr(emitter, "visible_length_is_physical"):
        if not emitter.visible_length_is_physical():
            return 1e30
    elif getattr(emitter, "last_visible_length_exceeds_range", False):
        return 1e30

    s = emitter.get_emission_points(p_locations, init_ke)

    exp_pes, exp_ts = emitter.get_expected_pes_ts(
        WCD,
        s,
        p_locations,
        direction_zs,
        mpmt_types,
        obs_pes,
    )

    nll = evaluate_pmt_nll(exp_pes, obs_pes, exp_ts, obs_ts, pmt_model)

    if not np.isfinite(nll):
        return 1e30

    if USE_TIMING_LIKELIHOOD and USE_T0_PRIOR:
        sigma_t0 = get_t0_prior_sigma(obs_pes, obs_ts)
        nll += abs(0.5 * (t0 / sigma_t0) ** 2)

    return float(nll)


def select_best_initial_seed(
    fit_mode,
    obs_pes,
    obs_ts,
    init_param_sets,
    emitter_template,
    mpmt_types,
    p_locations,
    direction_zs,
    WCD,
    pmt_model,
    range_lookup,
):
    best_info = None
    seed_scan = []

    for i, seed in enumerate(init_param_sets):
        emitter = emitter_template.copy()

        fval = evaluate_neg_log_likelihood(
            fit_mode,
            obs_pes,
            obs_ts,
            emitter,
            mpmt_types,
            p_locations,
            direction_zs,
            WCD,
            pmt_model,
            range_lookup,
            seed,
        )

        if not np.isfinite(fval):
            fval = np.inf

        info = {
            "seed_index": int(i),
            "fval": float(fval),
            "params": dict(seed),
        }

        if best_info is None or fval < best_info["fval"]:
            best_info = info

        if RETURN_FULL_SEED_SCAN:
            seed_scan.append(info)

    if best_info is None or not np.isfinite(best_info["fval"]):
        raise RuntimeError("All seed FCNs were non-finite.")

    if RETURN_FULL_SEED_SCAN:
        seed_scan_sorted = sorted(seed_scan, key=lambda x: x["fval"])
    else:
        seed_scan_sorted = [best_info]

    return (
        dict(best_info["params"]),
        int(best_info["seed_index"]),
        float(best_info["fval"]),
        seed_scan_sorted[:RETURN_TOP_N_SEEDS] if not RETURN_FULL_SEED_SCAN and RETURN_TOP_N_SEEDS > 0 else seed_scan_sorted,
    )


# =============================================================================
# 9. Minuit and prediction helpers
# =============================================================================
def make_minuit_for_single_event(
    fit_mode,
    obs_pes,
    obs_ts,
    start_params,
    emitter_template,
    mpmt_types,
    p_locations,
    direction_zs,
    WCD,
    pmt_model,
    range_lookup,
):
    fit_mode = canonical_fit_mode(fit_mode)
    emitter = emitter_template.copy()
    max_range = float(range_lookup.overall_distances_mm[-1])

    if fit_mode == "absorption":

        def nll(x0, y0, z0, cx, cy, visible_length, full_range, t0):
            params = {
                "x0": x0,
                "y0": y0,
                "z0": z0,
                "cx": cx,
                "cy": cy,
                "visible_length": visible_length,
                "full_range": full_range,
                "t0": t0,
            }
            return evaluate_neg_log_likelihood(
                fit_mode,
                obs_pes,
                obs_ts,
                emitter,
                mpmt_types,
                p_locations,
                direction_zs,
                WCD,
                pmt_model,
                range_lookup,
                params,
            )

        m = Minuit(nll, **start_params)
        m.limits["visible_length"] = (0.0, max_range)
        m.limits["full_range"] = (1.0, max_range)
        m.errors["visible_length"] = 60.0
        m.errors["full_range"] = 100.0

    else:

        def nll(x0, y0, z0, cx, cy, length, t0):
            params = {
                "x0": x0,
                "y0": y0,
                "z0": z0,
                "cx": cx,
                "cy": cy,
                "length": length,
                "t0": t0,
            }
            return evaluate_neg_log_likelihood(
                fit_mode,
                obs_pes,
                obs_ts,
                emitter,
                mpmt_types,
                p_locations,
                direction_zs,
                WCD,
                pmt_model,
                range_lookup,
                params,
            )

        m = Minuit(nll, **start_params)
        m.limits["length"] = (0.0, max_range)
        m.errors["length"] = 60.0

    m.limits["x0"] = (-2000.0, 2000.0)
    m.limits["y0"] = (-2000.0, 2000.0)
    m.limits["z0"] = (-2000.0, 2000.0)
    m.limits["cx"] = (-0.5, 0.5)
    m.limits["cy"] = (-0.5, 0.5)
    m.limits["t0"] = T0_LIMITS

    m.errors["x0"] = 30.0
    m.errors["y0"] = 30.0
    m.errors["z0"] = 30.0
    m.errors["cx"] = 0.01
    m.errors["cy"] = 0.01
    m.errors["t0"] = 0.1

    if not USE_TIMING_LIKELIHOOD:
        m.fixed["t0"] = True

    m.errordef = Minuit.LIKELIHOOD
    m.strategy = M_STRAT

    return m


def run_minuit(m):
    if USE_MIGRAD_FIRST:
        m.strategy = 0
        m.migrad(ncall=max(2000, int(0.35 * NCALL_MIGRAD)))

        bad = (m.fval is None) or (not np.isfinite(m.fval))
        if bad:
            m.simplex(ncall=max(2000, int(0.25 * NCALL_SIMPLEX)))
            m.strategy = M_STRAT
            m.migrad(ncall=NCALL_MIGRAD)
    else:
        m.strategy = M_STRAT
        m.simplex(ncall=NCALL_SIMPLEX)
        m.migrad(ncall=NCALL_MIGRAD)

    return m


def params_from_values_for_prediction(fit_mode, values):
    fit_mode = canonical_fit_mode(fit_mode)
    values = dict(values)

    params = {
        "x0": float(values["x0"]),
        "y0": float(values["y0"]),
        "z0": float(values["z0"]),
        "cx": float(values["cx"]),
        "cy": float(values["cy"]),
        "t0": float(values.get("t0", 0.0)),
    }

    if fit_mode == "absorption":
        params["visible_length"] = float(values["visible_length"])
        params["full_range"] = float(values["full_range"])
    else:
        params["length"] = float(values["length"])

    return params


def predict_expected_pes_ts(
    fit_mode,
    WCD,
    emitter_template,
    mpmt_types,
    p_locations,
    direction_zs,
    pmt_ids,
    obs_pes_for_norm,
    range_lookup,
    params,
):
    fit_mode = canonical_fit_mode(fit_mode)
    emitter = emitter_template.copy()
    params = dict(params)

    x0 = float(params["x0"])
    y0 = float(params["y0"])
    z0 = float(params["z0"])
    cx = float(params["cx"])
    cy = float(params["cy"])
    t0 = float(params.get("t0", 0.0))

    emitter.start_coord = (x0, y0, z0)
    emitter.starting_time = t0
    emitter.direction = unit_direction_from_cxcy(cx, cy)

    if fit_mode == "absorption":
        visible_length = float(params["visible_length"])
        full_range = float(params["full_range"])

        if visible_length > full_range:
            raise ValueError("visible_length cannot exceed full_range")

        ke0 = float(range_lookup.range_mm_to_energy(full_range))
        emitter.fixed_initial_KE = ke0
        init_ke = emitter.refresh_kinematics_from_length(visible_length)

        length_mm = visible_length

    else:
        length_mm = float(params["length"])
        visible_length = length_mm
        full_range = length_mm
        ke0 = float(range_lookup.range_mm_to_energy(full_range))
        emitter.fixed_initial_KE = None
        init_ke = emitter.refresh_kinematics_from_length(length_mm)

    if hasattr(emitter, "visible_length_is_physical"):
        if not emitter.visible_length_is_physical():
            raise ValueError("Requested visible length is not physical.")
    elif getattr(emitter, "last_visible_length_exceeds_range", False):
        raise ValueError("Requested visible length exceeds range-to-threshold.")

    s = emitter.get_emission_points(p_locations, init_ke)

    exp_pes, exp_ts = emitter.get_expected_pes_ts(
        WCD,
        s,
        p_locations,
        direction_zs,
        mpmt_types,
        obs_pes_for_norm,
    )

    meta = {
        "particle": emitter.particle_name,
        "fit_mode": fit_mode,
        "ke0_mev": float(ke0),
        "length_mm": float(length_mm),
        "visible_length_mm": float(visible_length),
        "full_range_mm": float(full_range),
        "range_to_threshold_mm": float(getattr(emitter, "range_to_threshold_mm", full_range)),
        "pmt_ids": np.asarray(pmt_ids, dtype=int),
        "s": s,
        "emitter": emitter,
        "components": getattr(emitter, "_last_expected_components", None),
    }

    return (
        np.asarray(pmt_ids, dtype=int),
        np.asarray(exp_pes, dtype=np.float64),
        np.asarray(exp_ts, dtype=np.float64),
        meta,
    )


# =============================================================================
# 10. Main single-event fitting function
# =============================================================================
def fit_single_event(
    event_array,
    *,
    fit_particle=None,
    fit_mode=None,
    config_root_file=None,
    return_seed_scan=None,
):
    particle = canonical_particle_name(FIT_PARTICLE if fit_particle is None else fit_particle)
    mode = canonical_fit_mode(FIT_MODE if fit_mode is None else fit_mode)

    set_active_particle(particle)

    if return_seed_scan is None:
        return_seed_scan = RETURN_FULL_SEED_SCAN

    print("Particle:", particle)
    print("Fit mode:", mode)

    if mode == "absorption":
        print("Fit parameters: x0, y0, z0, cx, cy, visible_length, full_range, t0")
    else:
        print("Fit parameters: x0, y0, z0, cx, cy, length, t0")

    range_lookup = ParticleRangeLookup(particle, table_dirs=[str(TABLES_DIR)])

    print("Range table:")
    print("  min KE [MeV]:", float(range_lookup.initial_energies_mev[0]))
    print("  max KE [MeV]:", float(range_lookup.initial_energies_mev[-1]))
    print("  max range [mm]:", float(range_lookup.overall_distances_mm[-1]))

    hall = Device.open_file(str(GEOMETRY_FILE))
    WCD = hall.wcds[0]

    good_wcte_pmts_set = load_good_wcte_pmts(
        CONFIG_ROOT_FILE if config_root_file is None else config_root_file
    )

    mpmt_info = load_mpmt_info()

    event_for_fit = np.asarray(event_array, dtype=np.float64)
    if APPLY_PEAK_TIME_WINDOW:
        event_for_fit = apply_peak_time_window_to_hit_array(event_for_fit)

    ev, active_pmt_ids_from_event = hit_array_to_event(
        event_for_fit,
        WCD,
        good_wcte_pmts_set,
        n_mpmt_total=106,
        shift_times=SHIFT_TIMES,
        n_earliest_for_t0=N_EARLIEST_FOR_T0,
    )

    p_locations, direction_zs, mpmt_slots, pmt_ids = get_pmt_placements_with_ids(
        ev,
        WCD,
        PMT_PLACEMENT,
    )

    obs_pes, obs_ts = build_observables_from_event(ev, pe_scale=PE_SCALE)

    if len(obs_pes) != len(pmt_ids):
        raise RuntimeError(
            f"Observable/geometry mismatch: len(obs_pes)={len(obs_pes)}, len(pmt_ids)={len(pmt_ids)}"
        )

    ring_keep_mask = np.isin(mpmt_slots, ALL_RING)

    obs_pes, obs_ts = apply_ring_mask_to_observables(
        obs_pes,
        obs_ts,
        ring_keep_mask,
        mode=RING_MASK_MODE,
    )

    mpmt_types = get_mpmt_slot_type(mpmt_slots, mpmt_info)

    print("Number of PMTs in fit:", len(obs_pes))
    print("Number of hit PMTs:", int(np.sum(obs_pes > 0)))
    print("Total observed PE:", float(np.sum(obs_pes)))

    initial_ke_seed = float(
        range_lookup.range_mm_to_energy(
            min(1000.0, float(range_lookup.overall_distances_mm[-1]))
        )
    )

    emitter_template = Emitter(
        0.0,
        (0.0, 0.0, 0.0),
        (0.0, 0.0, 1.0),
        0.96,
        500.0,
        18.0,
        particle=particle,
        track_end_mode=emitter_track_end_mode_for_fit_mode(mode),
        fixed_initial_KE=initial_ke_seed if mode == "absorption" else None,
    )

    delta_pdf_path = TABLES_DIR / "delta_e_angular_pdf_table.npz"
    if delta_pdf_path.exists() and hasattr(emitter_template, "load_delta_e_angular_pdf_table"):
        emitter_template.load_delta_e_angular_pdf_table(str(delta_pdf_path))

    pmt_model = PMT(1.0, 0.3, 1.0, 40.0, 0.2, 0.0)

    init_param_sets = build_fast_seed_grid(range_lookup, mode)
    print("Number of seed points:", len(init_param_sets))

    # Temporarily control whether the full seed scan is returned.
    #global RETURN_FULL_SEED_SCAN
    old_return_full_seed_scan = RETURN_FULL_SEED_SCAN
    RETURN_FULL_SEED_SCAN = bool(return_seed_scan)

    try:
        best_seed, best_seed_idx, best_seed_fval, seed_scan_sorted = select_best_initial_seed(
            mode,
            obs_pes,
            obs_ts,
            init_param_sets,
            emitter_template,
            mpmt_types,
            p_locations,
            direction_zs,
            WCD,
            pmt_model,
            range_lookup,
        )
    finally:
        RETURN_FULL_SEED_SCAN = old_return_full_seed_scan

    print("Best seed index:", best_seed_idx)
    print("Best seed FCN:", best_seed_fval)
    print("Best seed:", best_seed)

    m = make_minuit_for_single_event(
        mode,
        obs_pes,
        obs_ts,
        best_seed,
        emitter_template,
        mpmt_types,
        p_locations,
        direction_zs,
        WCD,
        pmt_model,
        range_lookup,
    )

    m = run_minuit(m)

    values = m.values.to_dict()

    if mode == "absorption":
        visible_length_mm = float(values["visible_length"])
        full_range_mm = float(values["full_range"])
        length_mm = visible_length_mm
    else:
        length_mm = float(values["length"])
        visible_length_mm = length_mm
        full_range_mm = length_mm

    ke0_mev = float(range_lookup.range_mm_to_energy(full_range_mm))

    params_for_prediction = params_from_values_for_prediction(mode, values)

    pmt_ids_pred, exp_pes, exp_ts, prediction_meta = predict_expected_pes_ts(
        mode,
        WCD,
        emitter_template,
        mpmt_types,
        p_locations,
        direction_zs,
        pmt_ids,
        obs_pes,
        range_lookup,
        params_for_prediction,
    )

    result = {
        "values": values,
        "errors": m.errors.to_dict(),
        "fval": float(m.fval) if m.fval is not None else np.nan,
        "valid": bool(m.valid),
        "edm": (
            float(m.fmin.edm)
            if getattr(m, "fmin", None) is not None and m.fmin is not None
            else np.nan
        ),

        "particle": particle,
        "fit_mode": mode,

        "length_mm": length_mm,
        "visible_length_mm": visible_length_mm,
        "full_range_mm": full_range_mm,
        "ke0_mev": ke0_mev,

        "best_seed_idx": best_seed_idx,
        "best_seed_fval": best_seed_fval,
        "best_seed": best_seed,
        "seed_scan": seed_scan_sorted if return_seed_scan else seed_scan_sorted[:RETURN_TOP_N_SEEDS],

        "obs_pes": obs_pes,
        "obs_ts": obs_ts,
        "pmt_ids": pmt_ids_pred,
        "exp_pes": exp_pes,
        "exp_ts": exp_ts,

        "mpmt_slots": mpmt_slots,
        "mpmt_types": mpmt_types,
        "p_locations": p_locations,
        "direction_zs": direction_zs,
        "ring_keep_mask": ring_keep_mask,

        "prediction_meta": prediction_meta,
        "event": ev,
        "event_array_used": event_for_fit,
        "WCD": WCD,
        "range_lookup": range_lookup,
        "pmt_model": pmt_model,
        "emitter_template": emitter_template,
        "minuit": m,
    }

    print("\nFit result:")
    print("  length         [mm]:", result["length_mm"])
    print("  visible_length [mm]:", result["visible_length_mm"])
    print("  full_range     [mm]:", result["full_range_mm"])
    print("  ke0            [MeV]:", result["ke0_mev"])
    print("  FCN:", result["fval"])
    print("  EDM:", result["edm"])
    print("  valid:", result["valid"])

    return result


# =============================================================================
# 11. Convenience helpers for accessing predictions
# =============================================================================
def prediction_array_from_result(result):
    """
    Return an array with columns:
        PMT ID, expected PE, expected time, observed PE, observed time
    """
    return np.column_stack([
        np.asarray(result["pmt_ids"], dtype=np.float64),
        np.asarray(result["exp_pes"], dtype=np.float64),
        np.asarray(result["exp_ts"], dtype=np.float64),
        np.asarray(result["obs_pes"], dtype=np.float64),
        np.asarray(result["obs_ts"], dtype=np.float64),
    ])


def get_expected_for_custom_params(result, params):
    """
    Recompute expected PEs/times for custom parameters using the same event geometry.

    For full_length mode, params should include:
        x0, y0, z0, cx, cy, length, t0

    For absorption mode, params should include:
        x0, y0, z0, cx, cy, visible_length, full_range, t0
    """
    return predict_expected_pes_ts(
        result["fit_mode"],
        result["WCD"],
        result["emitter_template"],
        result["mpmt_types"],
        result["p_locations"],
        result["direction_zs"],
        result["pmt_ids"],
        result["obs_pes"],
        result["range_lookup"],
        params,
    )


# =============================================================================
# 12. Usage examples
# =============================================================================

# Example 1: fit one event array in full_length mode.
# FIT_PARTICLE = "muon"
# FIT_MODE = "full_length"
# result = fit_single_event(event_array)

# Example 2: fit one event array in absorption mode.
# FIT_PARTICLE = "proton"
# FIT_MODE = "absorption"
# result = fit_single_event(event_array)

# Access expected PEs and PMT IDs:
# pmt_ids = result["pmt_ids"]
# exp_pes = result["exp_pes"]
# exp_ts = result["exp_ts"]

# Build a compact table:
# pred = prediction_array_from_result(result)
# pred[:10]

# Columns:
#   pred[:, 0] = PMT ID
#   pred[:, 1] = expected PE
#   pred[:, 2] = expected time
#   pred[:, 3] = observed PE
#   pred[:, 4] = observed time

In [3]:
# =============================================================================
# Run-based event loading, matching scripts/batch_fit_driver.py
# =============================================================================

from event_loader import get_selected_events


# -----------------------------------------------------------------------------
# Run/event-selection settings
# -----------------------------------------------------------------------------
RUN = 2079
N_EVENTS_TO_LOAD = 1000

PRODUCTION_VERSION = "production_v1_0"

# This should usually match FIT_PARTICLE, but you can override it if needed.
PARTICLE_SELECTION_LABEL = FIT_PARTICLE

# Keep the historical ±0.2 ns TOF window.
SELECTION_TOF_NS = None          # None means event_loader uses tof_mean_<particle>
SELECTION_TOF_WINDOW_NS = 0.2
SELECTION_TOF_FIELD = None       # optional override, e.g. "tof_mean_proton"
SELECTION_MOMENTUM_FIELD = None  # optional override

# Keep this as in the batch driver unless you intentionally change the T5 selection.
SELECTION_T5_PARTICLE_NR = 1

USE_PEAK_TIME_CUT = True
PEAK_WINDOW_NS = 100.0
PEAK_BIN_WIDTH_NS = 50.0


def default_config_root_file(run, production_version=PRODUCTION_VERSION):
    return (
        f"/eos/experiment/wcte/data/2025_commissioning/processed_offline_data/"
        f"{production_version}/{int(run)}/WCTE_merged_production_R{int(run)}.root"
    )


def load_events_for_run(
    run,
    n_events,
    *,
    particle=None,
    root_file=None,
):
    """
    Load selected events for one run using the same event_loader path as the
    real-data batch driver.

    Returns
    -------
    events : list
        Each event is an array with columns [pmt_id, charge, time].
    root_file : str
        The ROOT file used for both event loading and GOOD_WCTE_PMTS.
    """
    particle = canonical_particle_name(FIT_PARTICLE if particle is None else particle)

    if root_file is None:
        root_file = default_config_root_file(run)

    events = get_selected_events(
        int(run),
        int(n_events),
        particle=particle,
        root_file=root_file,
        use_peak_time_cut=USE_PEAK_TIME_CUT,
        peak_window=PEAK_WINDOW_NS,
        peak_bin_width=PEAK_BIN_WIDTH_NS,
        tof_primary=SELECTION_TOF_NS,
        tof_window=SELECTION_TOF_WINDOW_NS,
        tof_scalar_field=SELECTION_TOF_FIELD,
        momentum_scalar_field=SELECTION_MOMENTUM_FIELD,
        t5_particle_nr=SELECTION_T5_PARTICLE_NR,
    )

    print(f"Loaded {len(events)} selected events from run {run}")
    print("ROOT file:", root_file)

    return events, root_file


def fit_single_event_from_run(
    run,
    event_index=0,
    *,
    n_events=N_EVENTS_TO_LOAD,
    fit_particle=None,
    fit_mode=None,
    root_file=None,
    return_loaded_events=False,
):
    """
    Load selected events from a run, then fit one event by index.

    Example
    -------
    result = fit_single_event_from_run(
        run=2079,
        event_index=0,
        n_events=1000,
        fit_particle="muon",
        fit_mode="full_length",
    )
    """
    particle = canonical_particle_name(FIT_PARTICLE if fit_particle is None else fit_particle)

    events, root_file = load_events_for_run(
        run,
        n_events,
        particle=particle,
        root_file=root_file,
    )

    if len(events) == 0:
        raise RuntimeError(f"No selected events were loaded for run {run}.")

    if event_index < 0 or event_index >= len(events):
        raise IndexError(
            f"event_index={event_index} is out of range for {len(events)} loaded events."
        )

    result = fit_single_event(
        events[event_index],
        fit_particle=particle,
        fit_mode=FIT_MODE if fit_mode is None else fit_mode,
        config_root_file=root_file,
    )

    result["run"] = int(run)
    result["event_index_in_selected_events"] = int(event_index)
    result["config_root_file"] = root_file

    if return_loaded_events:
        return result, events

    return result

In [ ]:
FIT_PARTICLE = "muon"
FIT_MODE = "full_length"

result = fit_single_event_from_run(
    run=2079,
    event_index=0,
    n_events=100,
)

pmt_ids = result["pmt_ids"]
exp_pes = result["exp_pes"]
exp_ts = result["exp_ts"]

pred = prediction_array_from_result(result)
pred[:10]

Selected-event loader
---------------------
Particle label:              muon
ROOT file:                   /eos/experiment/wcte/data/2025_commissioning/processed_offline_data/production_v1_0/2079/WCTE_merged_production_R2079.root
T5 particle nr:              1
TOF mean [ns]:               15.0985
TOF window [ns]:             +/- 0.2
Beam momentum after window:  396.51355408819654


Peak-time calibration
---------------------
ROOT entries requested:        100
ROOT entries scanned:          5
Selected events used:          50
Estimated median peak time:    2175.00 ns

